In [1]:
import polars as pl
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error, root_mean_squared_error

from catboost import CatBoostRegressor

import optuna

import nbformat


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


from transformers import AutoTokenizer, AutoModel

import warnings

warnings.filterwarnings('ignore')

/Users/egor/VS_GIT_repositories/BYTE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet('/Users/egor/VS_GIT_repositories/BYTE/src/data/tabular/yandex_realty_cleaned.parquet')
df['price_per_m2'] = df['price_numeric'] / df['area']
df['description'] = df['description'].apply(lambda x: str(x))
df['address'] = df['address'].apply(lambda x: str(x))
df['area'] = df['area'].fillna(0)
df.shape

(93444, 22)

In [3]:
df.head()

,offer_id,price,price_numeric,old_price,area,rooms,floor,price_per_m2,metro,metro_time,...,main_image,photo_count,badges,publish_date,url,title,description,image_urls,self_floor,max_floor
0,7035113340557126091,7 500 000 ₽,7500000.0,NaN,17.7,студия,9 этаж из 16,423728.813559,Калитники,9.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,None,https://realty.yandex.ru/offer/703511334055712...,апартаменты-студия,Номер лота: 99696. Панорамный вид из больших о...,https://avatars.mds.yandex.net/get-realty-offe...,16.0,16.0
1,7035113340416809607,7 500 000 ₽,7500000.0,NaN,17.0,студия,2 этаж из 2,441176.470588,Соколиная гора,8.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/703511334041680...,апартаменты-студия,Номер лота: 87440. Продается студия с дизайнер...,https://avatars.mds.yandex.net/get-realty-offe...,2.0,2.0
2,7053956964805047621,12 200 000 ₽,12200000.0,NaN,17.9,студия,2 этаж из 48,681564.245810,Тушинская,10.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,3 квартал 2027,https://realty.yandex.ru/offer/705395696480504...,квартира-студия,"Арт. 119802099 Студия 17,9 м в CITYZEN Урбан-б...",https://avatars.mds.yandex.net/get-realty-offe...,48.0,48.0
3,7053956914445503237,7 300 000 ₽,7300000.0,NaN,15.7,студия,5 этаж из 5,464968.152866,Бутырская,17.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,6 часов назад назад,https://realty.yandex.ru/offer/705395691444550...,квартира-студия,Арт. 134010491 СПЕЦИАЛЬНО для наших клиентов с...,https://avatars.mds.yandex.net/get-realty-offe...,5.0,5.0
4,3699730400767130013,10 802 031 ₽,10802031.0,NaN,14.1,студия,5 этаж из 16,766101.489362,Коммунарка,14.0,...,https://avatars.mds.yandex.net/get-realty-offe...,1.0,None,2 квартал 2026,https://realty.yandex.ru/offer/369973040076713...,квартира-студия,Строим кварталы для жизни с заботой о будущем....,https://avatars.mds.yandex.net/get-realty-offe...,16.0,16.0


In [4]:
def get_encoders(df, columns=['rooms', 'metro', 'title', 'self_floor', 'max_floor']):
    label_encoders = {}
    for col in columns:
        enc = LabelEncoder()
        enc.fit(df[col])
        label_encoders[col] = enc
    return label_encoders

def apply_encoders(df, encoders, columns=['rooms', 'metro', 'title', 'self_floor', 'max_floor']):
    df_cp = df.copy()
    for col in columns:
        df_cp[col] = encoders[col].transform(df_cp[col] )
    return df_cp

def scale(df, columns=['price_numeric','area', 'price_per_m2']):
    scaler = StandardScaler()
    scaler.fit(df[columns])
    df[columns] = scaler.transform(df[columns])
    return scaler

In [5]:
class RoualtyDataset(Dataset):
    def __init__(self, df, tokenizer):

        self.df = df
        self.tokenizer = tokenizer

        self.num_features = ['area']
        self.cat_features = ['rooms', 'metro', 'title', 'self_floor', 'max_floor']


    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, idx):

        item = self.df.iloc[idx]
        
        numeric = item[self.num_features]
        category = item[self.cat_features]


        # description_text = f"Описание: {item['description']}"
        # address_text = f"Адрес: {item['address']}"

        # description = self.tokenizer(description_text, padding='max_length', truncation=True, 
        # max_length=256, return_tensors='pt')['input_ids'].squeeze(0)
        # address = self.tokenizer(address_text, padding='max_length', truncation=True, 
        # max_length=256, return_tensors='pt')['input_ids'].squeeze(0)

        description = torch.tensor(item['desc_ids'], dtype=torch.long)
        address = torch.tensor(item['addr_ids'], dtype=torch.long)

        return {'num_featues': torch.tensor(numeric, dtype=torch.float32),
                'cat_features':torch.tensor(category, dtype=torch.long) ,
                'description':description,
                'address':address,
                'target':torch.tensor(item['price_numeric'], dtype=torch.float32)
                }

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def precompute_rope_frequencies(dim, seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(seq_len)
    freqs = torch.outer(t, freqs).float()
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    
    rotated_x = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return rotated_x.flatten(-2)

class RoPETransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        # Feed-Forward
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.SiLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, cos, sin):
        residual = x
        x = self.norm1(x)
        
        batch_size, seq_len, _ = x.shape

        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        cos_input = cos[:seq_len, :].view(1, 1, seq_len, -1)
        sin_input = sin[:seq_len, :].view(1, 1, seq_len, -1)
        
        q = apply_rope(q, cos_input, sin_input)
        k = apply_rope(k, cos_input, sin_input)
        
        attn_output = F.scaled_dot_product_attention(q, k, v)
        attn_output = attn_output.transpose(1, 2).reshape(batch_size, seq_len, self.embed_dim)
        
        x = residual + self.out_proj(attn_output)

        x = x + self.ffn(self.norm2(x))
        
        return x

In [15]:
import torch
import torch.nn as nn

class RoyaltyModel(nn.Module):
    def __init__(self, text_encoder, emb_dim=128, emb_cnt=[13, 349, 37, 78, 78], n_blocks=3):
        super().__init__()
        self.text_encoder = text_encoder
        self.emb_dim = emb_dim

        for param in text_encoder.parameters():
            param.requires_grad = False 
        
        self.embeddings = nn.ModuleList([
            nn.Embedding(cnt, emb_dim) for cnt in emb_cnt
        ])
    
        self.desc_mlp = nn.Sequential(
            nn.Linear(312, 128), nn.SiLU(),
            nn.Linear(128, emb_dim), nn.SiLU()
        )

        self.address_mlp = nn.Sequential(
            nn.Linear(312, 128), nn.SiLU(),
            nn.Linear(128, emb_dim), nn.SiLU()
        )
    
        self.area_mlp = nn.Sequential(
            nn.Linear(1, 64), nn.SiLU(),
            nn.Linear(64, emb_dim)
        )
        
        self.transformers = nn.ModuleList([
            RoPETransformerBlock(embed_dim=emb_dim, num_heads=8, ff_dim=512, dropout=0.1) 
            for _ in range(n_blocks)
        ])

        self.max_seq_len = 2 + 1 + len(emb_cnt) # desc + addr + area + cats = 8
        cos, sin = precompute_rope_frequencies(emb_dim // 8, self.max_seq_len)
        self.register_buffer("rope_cos", cos)
        self.register_buffer("rope_sin", sin)

        self.head = nn.Sequential(
            nn.Linear(emb_dim, 128), nn.SiLU(),
            nn.Linear(128, 1)
        )

    def forward(self, numeric, category, description, address):
        t_desc = self.text_encoder(description).last_hidden_state[:, 0, :]
        t_addr = self.text_encoder(address).last_hidden_state[:, 0, :]
        
        desc_emb = self.desc_mlp(t_desc).unsqueeze(1) 
        addr_emb = self.address_mlp(t_addr).unsqueeze(1)
        area_emb = self.area_mlp(numeric).unsqueeze(1) 

        cat_embs = []
        for i, emb_layer in enumerate(self.embeddings):
            cat_embs.append(emb_layer(category[:, i]).unsqueeze(1))
        
        cat_embs = torch.cat(cat_embs, dim=1) # (B, 5, emb_dim)

        x = torch.cat([desc_emb, addr_emb, area_emb, cat_embs], dim=1) # (B, 8, emb_dim)

        for block in self.transformers:
            x = block(x, self.rope_cos, self.rope_sin)

        x = x.mean(dim=1)
        return self.head(x)


In [8]:
cat_columns = ['rooms', 'metro', 'title', 'self_floor', 'max_floor']
encoders = get_encoders(df, columns=['rooms', 'metro', 'title', 'self_floor', 'max_floor'])
enc_df = apply_encoders(df, encoders, columns=['rooms', 'metro', 'title', 'self_floor', 'max_floor'])

emb_cnt = {col:enc_df[col].nunique()+1 for col in cat_columns}
emb_cnt

{'rooms': 13, 'metro': 349, 'title': 37, 'self_floor': 78, 'max_floor': 78}

In [9]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
text_encoder = AutoModel.from_pretrained("cointegrated/rubert-tiny2")

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 2214.16it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# Один раз перед созданием Dataset
def tokenize_texts(texts, tokenizer, max_len=256):
    encodings = tokenizer(
        texts.tolist(), 
        padding='max_length', 
        truncation=True, 
        max_length=max_len, 
        return_tensors='pt'
    )
    return encodings['input_ids']

desc_ids = tokenize_texts(enc_df['description'], tokenizer)
addr_ids = tokenize_texts(enc_df['address'], tokenizer)

enc_df['desc_ids'] = list(desc_ids.numpy())
enc_df['addr_ids'] = list(addr_ids.numpy())

enc_df['price_numeric'] = np.log1p(enc_df['price_numeric'])

In [11]:
train, test = train_test_split(enc_df, test_size=0.5, shuffle=True)

train, val = train_test_split(train, test_size=0.2, shuffle=True)

# train, val = train_test_split(enc_df, test_size=0.2, shuffle=True)



# scaler = scale(train, columns=['price_numeric'])

train_dataset = RoualtyDataset(train, tokenizer=tokenizer)
val_dataset = RoualtyDataset(val, tokenizer=tokenizer)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)


In [12]:
batch = next(iter(train_dataset))

In [16]:
model = RoyaltyModel(text_encoder, emb_dim=128, n_blocks=3)

print(f"Total_params = {sum(p.numel() for p in model.parameters()) / 1024**3} GB")

Total_params = 0.02793768886476755 GB


In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.MSELoss() 
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

epochs = 10

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    
    for batch in train_loop:
        numeric = batch['num_featues'].to(device).float()
        category = batch['cat_features'].to(device).long()
        target = batch['target'].to(device).float()
        description = batch['description'].to(device)
        address = batch['address'].to(device)

        optimizer.zero_grad()
        outputs = model(numeric, category, description, address).squeeze()
        
        loss = criterion(outputs, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)
    
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")

    with torch.no_grad():
        for batch in val_loop:
            numeric = batch['num_featues'].to(device).float()
            category = batch['cat_features'].to(device).long()
            target = batch['target'].to(device).float()
            description = batch['description'].to(device)
            address = batch['address'].to(device)

            outputs = model(numeric, category, description, address).squeeze()
            
            loss = criterion(outputs, target)
            val_loss += loss.item()
            
            val_loop.set_postfix(loss=loss.item())

            all_preds.extend(outputs.cpu().detach().numpy())
            all_targets.extend(target.cpu().detach().numpy())

    all_preds = np.expm1(np.array(all_preds).reshape(-1, 1))
    all_targets = np.expm1(np.array(all_targets).reshape(-1, 1))

    avg_val_loss = val_loss / len(val_loader)
    r2 = r2_score(all_targets, all_preds)
    mae = mean_absolute_error(all_targets, all_preds)
    mape = mean_absolute_percentage_error(all_targets, all_preds)

    scheduler.step() 

    print(f"\n[Epoch {epoch+1}] Results:")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val MSE:    {avg_val_loss:.4f}")
    print(f"  Val MAE:    {mae:.4f}")
    print(f"  Val R2:     {r2:.4f}")
    print(f"  Val MAPE:   {mape:.2%}")
    print(f"  Current LR: {optimizer.param_groups[0]['lr']:.6f}")
    print("-" * 30)


Epoch 1/10 [Train]:   0%|          | 0/293 [00:00<?, ?it/s]

Epoch 1/10 [Val]: 100%|██████████| 94/94 [02:29<00:00,  1.59s/it, loss=0.0785]



[Epoch 1] Results:
  Train Loss: 25.1019
  Val MSE:    0.1703
  Val MAE:    12454002.0000
  Val R2:     0.4895
  Val MAPE:   28.94%
  Current LR: 0.000098
------------------------------


Epoch 2/10 [Train]:  27%|██▋       | 79/293 [23:47<1:06:49, 18.74s/it, loss=0.0912]